<a href="https://colab.research.google.com/github/roshanis/notebooks/blob/main/AI_Engineer_World's_Fair_Arize_101_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started with Arize AX

**Hands-on workshop companion notebook**

This is the runnable notebook for the *Getting Started with Arize AX* workshop. Over the next ~100 minutes we'll build a real AI agent and put it through the full evaluation lifecycle in Arize AX:

1. **Set up tracing** — instrument a Claude agent with Arize AX
2. **Build an agent** — a financial analysis chatbot with web search
3. **Generate test data** — a handful of diverse runs to evaluate
4. **Look at your data** — error analysis *before* writing any evals
5. **Write a code eval** — a deterministic ticker-mention check
6. **Run built-in evals** — correctness vs. faithfulness against the agent's research
7. **Write a custom rubric** — an actionability eval from scratch
8. **Meta-evaluate** — test the judge against human labels you apply in the AX UI
9. **Improve the agent automatically** — feed the judge's explanations back to Claude, then run an experiment to prove the fix worked

Run it in **Google Colab**: choose *File → Save a copy in Drive* so you can edit your own version.

You'll need:
- An **Anthropic API key** ([console.anthropic.com](https://console.anthropic.com))
- An **Arize AX account** (start a free trial at [arize.com](https://arize.com))
  - Your **Arize Space ID** (in your AX workspace settings)
  - An **Arize API key** (generate one in Settings)

> Going to production after the workshop? Online evals, monitors, and the coding-agent improvement loop live in the AX UI and docs ([arize.com/docs/ax](https://arize.com/docs/ax)). The `arize-skills` plugin (`npx skills add Arize-ai/arize-skills`) lets a coding agent do most of this for you.

## Step 1: Set Up Tracing

Tracing captures a structured record of every step your agent takes — every LLM call, every tool invocation, every decision — with inputs and outputs at each point. We set it up once, and data flows automatically.

In [ ]:
%pip install claude-agent-sdk openinference-instrumentation-claude-agent-sdk arize arize-otel arize-phoenix anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pydantic[email] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.5/326.5 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.3/212.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.4/1

In [ ]:
import os
from google.colab import userdata

anthropic_api_key = userdata.get('anthropic-api-key')
arize_api_key = userdata.get('arize-api-key')
arize_space_id = userdata.get('arize-space-id')

os.environ["ANTHROPIC_API_KEY"] = anthropic_api_key
os.environ["ARIZE_API_KEY"] = arize_api_key
os.environ["ARIZE_SPACE_ID"] = arize_space_id

In [ ]:
from arize.otel import register, Endpoint
from openinference.instrumentation.claude_agent_sdk import ClaudeAgentSDKInstrumentor

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="aiewf-financial-demo",
    endpoint=Endpoint.ARIZE,
    batch=False
)

# Auto-instrument the Claude Agent SDK so every LLM and tool call
# becomes a span automatically.
ClaudeAgentSDKInstrumentor().instrument(tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: aiewf-financial-demo
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [ ]:
# Set up an ArizeClient for reading spans and logging evaluations.
from arize import ArizeClient

arize_client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])
SPACE_ID = os.environ["ARIZE_SPACE_ID"]
PROJECT_NAME = "aiewf-financial-demo"


## Step 2: Build the Agent

We're building a financial analysis chatbot using the Claude Agent SDK. The agent works in two turns:
- **Turn 1: Research** — searches the web for real financial data
- **Turn 2: Write** — compiles the research into a readable report

The `ClaudeSDKClient` maintains conversation context between turns, so the writer has full access to the researcher's findings.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

RESEARCH_PROMPT = """Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends."""

WRITE_PROMPT = """Now write a concise financial report based on your research above."""

options = ClaudeAgentOptions(
    model="claude-haiku-4-5-20251001",
    allowed_tools=["WebSearch"],
    permission_mode="acceptEdits",
)


async def financial_report(tickers: str, focus: str, verbose: bool = True) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research
            if verbose:
                print(f"--- Researching {tickers} ({focus}) ---")
            await client.query(RESEARCH_PROMPT.format(tickers=tickers, focus=focus))
            async for message in client.receive_response():
                if verbose and isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            preview = block.text[:200].replace("\n", " ")
                            print(f"  [research] {preview}...")

            # Turn 2: Write report
            if verbose:
                print(f"--- Writing report ---")
            await client.query(WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                            if verbose:
                                preview = block.text[:200].replace("\n", " ")
                                print(f"  [writing] {preview}...")
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
    tracer_provider.force_flush()
    return report


### Run the agent once to test

This will take a minute or two

In [ ]:
result = await financial_report("TSLA", "financial performance and growth outlook")
print(result)

--- Researching TSLA (financial performance and growth outlook) ---
  [research] ## Tesla (TSLA) Research: Financial Performance & Growth Outlook  ### Financial Performance Summary  #### Q1 2026 Results Tesla delivered solid Q1 2026 earnings with revenue reaching **$22.4 billion**...
--- Writing report ---
  [writing] # TESLA INC. (NASDAQ: TSLA) ## Financial Report – June 2026  ---  ## EXECUTIVE SUMMARY  Tesla reported Q1 2026 revenue of $22.4B (+16% YoY) with strong margin expansion to 21.1%, beating EPS expectati...
# TESLA INC. (NASDAQ: TSLA)
## Financial Report – June 2026

---

## EXECUTIVE SUMMARY

Tesla reported Q1 2026 revenue of $22.4B (+16% YoY) with strong margin expansion to 21.1%, beating EPS expectations. However, the company faces near-term challenges including declining vehicle deliveries (-9% YoY) and an aggressive $25B+ capital expenditure program that will generate negative free cash flow throughout 2026. Success hinges on executing autonomous driving and robotics in

## Step 3: Generate Test Data

To run meaningful evals, we need more than one trace. Here are 12 test queries covering different tickers, focus areas, and levels of complexity.

In [ ]:
test_queries = [
    {"tickers": "AAPL", "focus": "revenue growth and services segment"},
    {"tickers": "NVDA", "focus": "AI chip demand and valuation metrics"},
    {"tickers": "AMZN", "focus": "AWS performance and profitability"},
    {"tickers": "GOOGL", "focus": "advertising revenue and AI strategy"},
    {"tickers": "MSFT", "focus": "cloud computing segment"},
    {"tickers": "META", "focus": "metaverse investments and ad revenue"},
    {"tickers": "TSLA", "focus": "vehicle deliveries and margins"},
    {"tickers": "RIVN", "focus": "financial health and future growth"},
    {"tickers": "AAPL, MSFT", "focus": "comparative financial analysis"},
    {"tickers": "NVDA", "focus": "competitive landscape and market share"},
    {"tickers": "KO", "focus": "dividend yield and stability"},
    {"tickers": "AMZN", "focus": "profitability trends and outlook"},
]

In [ ]:
for i, query in enumerate(test_queries):
    print(f"[{i+1}/{len(test_queries)}] {query['tickers']} — {query['focus']}")
    await financial_report(query["tickers"], query["focus"], verbose=False)
    print(f"  done")

[1/12] AAPL — revenue growth and services segment
  done
[2/12] NVDA — AI chip demand and valuation metrics
  done
[3/12] AMZN — AWS performance and profitability
  done
[4/12] GOOGL — advertising revenue and AI strategy
  done
[5/12] MSFT — cloud computing segment
  done
[6/12] META — metaverse investments and ad revenue
  done
[7/12] TSLA — vehicle deliveries and margins
  done
[8/12] RIVN — financial health and future growth
  done
[9/12] AAPL, MSFT — comparative financial analysis
  done
[10/12] NVDA — competitive landscape and market share
  done
[11/12] KO — dividend yield and stability
  done
[12/12] AMZN — profitability trends and outlook
  done


## Step 4: Look at Your Data — Error Analysis

Before writing a single eval, read your traces. This is the step most tutorials skip — and it's the most important one. You need to understand *what's actually going wrong* before you can measure it.

We'll:
1. **Examine traces** — read the input and output of a few runs
2. **Categorize failures** — label each trace with a root cause
3. **Build a frequency table** — see where to focus first

In [ ]:
from datetime import datetime, timedelta

spans_df = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=datetime.utcnow() - timedelta(days=7),
    end_time=datetime.utcnow(),
)

# Filter to top-level agent spans (the full financial_report calls)
parent_spans = spans_df[spans_df["parent_id"].isna()].copy()

# AX export sometimes returns both top-level (input/output) and attributes.*
# versions of the same field. Only rename if the short name isn't already there,
# then drop any duplicates that snuck in.
if "input" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
if "output" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
parent_spans = parent_spans.loc[:, ~parent_spans.columns.duplicated()]

print(f"Found {len(parent_spans)} top-level spans")

  arize._exporter.client | INFO | Fetching data...


/tmp/ipykernel_3840/623352183.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow() - timedelta(days=7),
/tmp/ipykernel_3840/623352183.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time=datetime.utcnow(),
INFO:arize._exporter.client:Fetching data...
  exporting 207 rows: 100%|█████████████████████| 207/207 [00:00, 1304.65 row/s]


Found 13 top-level spans


### Examine a few traces

Read the input and a snippet of the output side by side. What jumps out? Look for: hallucinated numbers, missing comparisons, vague recommendations, data that "looks plausible" but you can't verify.

In [ ]:
# Show the first 4 traces: input query and a truncated output
for i, (idx, row) in enumerate(parent_spans.head(4).iterrows()):
    print(f"{'='*60}")
    print(f"TRACE {i+1}")
    print(f"  INPUT:  {row['input']}")
    print(f"  OUTPUT: {str(row['output'])[:300]}...")
    print()

TRACE 1
  INPUT:  Research: TSLA
Focus: financial performance and growth outlook
  OUTPUT: # TESLA INC. (NASDAQ: TSLA)
## Financial Report – June 2026

---

## EXECUTIVE SUMMARY

Tesla reported Q1 2026 revenue of $22.4B (+16% YoY) with strong margin expansion to 21.1%, beating EPS expectations. However, the company faces near-term challenges including declining vehicle deliveries (-9% YoY...

TRACE 2
  INPUT:  Research: AAPL
Focus: revenue growth and services segment
  OUTPUT: I'll present the financial report directly for you:

---

# **Apple Inc. (AAPL) Financial Report**
## Q2 2026 Results & Services Segment Analysis
**Report Date:** June 28, 2026

---

## **EXECUTIVE SUMMARY**

Apple delivered strong financial performance in fiscal 2026's first half with total revenue...

TRACE 3
  INPUT:  Research: NVDA
Focus: AI chip demand and valuation metrics
  OUTPUT: # NVIDIA Corporation (NVDA)
## Financial Analysis Report
**As of June 28, 2026**

---

## I. COMPANY OVERVIEW

NVIDIA is the d

### Categorize failures by root cause

Now label each trace with what you observe. Don't overthink the categories — "looks good," "hallucination," "reasoning gap," "missing data" are fine. The goal is to build intuition about *what kinds of things go wrong*.

In [ ]:
import pandas as pd

# Label each trace with a root cause category.
# Replace these with your own observations after reading the traces above!
trace_categories = {
    "TSLA — financial performance": "looks good",
    "NVDA — AI chip demand": "looks good",
    "AMZN — AWS performance": "possible hallucination",
    "GOOGL — advertising revenue": "looks good",
    "MSFT — cloud computing": "looks good",
    "META — metaverse investments": "reasoning gap",
    "TSLA — vehicle deliveries": "looks good",
    "RIVN — financial health": "unverifiable data",
    "AAPL, MSFT — comparative": "reasoning gap",
    "NVDA — competitive landscape": "possible hallucination",
    "KO — dividend yield": "missing recommendation",
    "AMZN — profitability trends": "possible hallucination",
}

categories_df = pd.DataFrame(
    list(trace_categories.items()),
    columns=["trace", "category"]
)
display(categories_df)

,trace,category
0,TSLA — financial performance,looks good
1,NVDA — AI chip demand,looks good
2,AMZN — AWS performance,possible hallucination
3,GOOGL — advertising revenue,looks good
4,MSFT — cloud computing,looks good
5,META — metaverse investments,reasoning gap
6,TSLA — vehicle deliveries,looks good
7,RIVN — financial health,unverifiable data
8,"AAPL, MSFT — comparative",reasoning gap
9,NVDA — competitive landscape,possible hallucination


### Frequency table — where should we focus?

Count the categories. The most frequent failure type is where you focus first. But frequency isn't everything — **frequency × severity = priority**. A hallucinated stock price is worse than an awkward sentence.

In [ ]:
# Frequency table of root cause categories
print("Root cause frequency:")
print("-" * 35)
counts = categories_df["category"].value_counts()
for category, count in counts.items():
    bar = "█" * count
    print(f"  {category:<25} {count}  {bar}")
print(f"\nTotal traces: {len(categories_df)}")
print(f"Failure rate: {(len(categories_df) - counts.get('looks good', 0)) / len(categories_df):.0%}")

Root cause frequency:
-----------------------------------
  looks good                5  █████
  possible hallucination    3  ███
  reasoning gap             2  ██
  unverifiable data         1  █
  missing recommendation    1  █

Total traces: 12
Failure rate: 58%


## Step 5: Code Eval — Ticker Mention Check

Code evals are deterministic functions — no LLM, no API call, no cost. Our simplest useful check: does the output actually mention the ticker we asked about?


1. Click + Add Online Evaluator
2. Give it a name e.g. MentionsEvaluator
3. In `Define imports`, paste this code:

```
from typing import Any, Optional
import re
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    CodeEvaluator,
)
```

4. In `Define Code Evaluator Class` paste this code:

```
class MentionsEvaluator(CodeEvaluator):
    def evaluate(
        self,
        *,
        query: Optional[str] = None,
        report: Optional[str] = None,
        **kwargs: Any,
    ) -> EvaluationResult:
        # Extract ticker symbols from the input (uppercase 1-5 letter words)
        tickers = re.findall(r"\b([A-Z]{1,5})\b", query)
        # Filter to likely tickers (skip common words)
        likely_tickers = [
            t
            for t in tickers
            if len(t) >= 2
            and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
        ]
    
        if not likely_tickers or not report:
            return EvaluationResult(
                label = "unknown",
                score = 0
            )
    
        missing = [t for t in likely_tickers if t not in report.upper()]
    
        if not missing:
            return EvaluationResult(
                label = "pass",
                score = 1
            )
        else:
            return EvaluationResult(
                label = "fail",
                score = 0
            )
```

5. Under `Matching Traces` select `Single trace`
6. Select the input and match it to the `query` variable
7. Select the output and match it to the `report` variable
8. Click Save


### Alternative path: code evals in code

If you don't want to use the UI, here's how you can create and run a code evaluator solely from the notebook. This is the same evaluator as the online evaluator.


In [ ]:
print(f"Using {len(parent_spans)} top-level spans for code evals")

Using 13 top-level spans for code evals


In [ ]:
import re
from phoenix.evals import create_evaluator
from openinference.instrumentation import suppress_tracing
from phoenix.evals import evaluate_dataframe


@create_evaluator(name="mentions_ticker", kind="code")
def mentions_ticker(input, output):
    """Code eval: does the output mention the ticker(s) we asked about?"""
    # Extract ticker symbols from the input (uppercase 1-5 letter words)
    tickers = re.findall(r"\b([A-Z]{1,5})\b", input)
    # Filter to likely tickers (skip common words)
    likely_tickers = [
        t
        for t in tickers
        if len(t) >= 2
        and t not in ("AI", "US", "CEO", "CFO", "IPO", "ETF", "AWS", "USE")
    ]

    if not likely_tickers or not output:
        return {"label": "unknown", "score": 0}

    missing = [t for t in likely_tickers if t not in output.upper()]

    if not missing:
        return {"label": "pass", "score": 1}
    else:
        return {
            "label": "fail",
            "score": 0,
            "explanation": f"Missing tickers: {', '.join(missing)}",
        }


with suppress_tracing():
    results = evaluate_dataframe(dataframe=parent_spans, evaluators=[mentions_ticker])

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
# Show all mentions_ticker results (not just head)
import pandas as pd

ticker_scores = pd.json_normalize(results["mentions_ticker_score"])
print("Mentions Ticker Results:")
print(f"  {ticker_scores['label'].value_counts().to_dict()}")
print(f"  {ticker_scores['label'].value_counts().get('pass', 0)}/{len(ticker_scores)} passed")

# Show any failures with explanations
failures = ticker_scores[ticker_scores["label"] == "fail"]
if len(failures) > 0:
    print(f"\nFailures:")
    for _, row in failures.iterrows():
        print(f"  {row.get('explanation', 'no explanation')}")
else:
    print("\nAll passed!")

Mentions Ticker Results:
  {'pass': 12, 'fail': 1}
  12/13 passed

Failures:
  Missing tickers: AMZN


Send the results back to Arize AX

In [ ]:
# Helper: convert evaluate_dataframe results to AX's expected eval format
# and log them so they appear as annotations on each trace.
#
# AX expects a dataframe with columns:
#   - context.span_id     (which span to attach to)
#   - eval.<name>.label
#   - eval.<name>.score
#   - eval.<name>.explanation
from phoenix.evals.utils import to_annotation_dataframe

def log_eval_to_ax(eval_results_df, eval_name):
    annotations = to_annotation_dataframe(dataframe=eval_results_df)
    # to_annotation_dataframe outputs columns:
    #   name, label, score, explanation, metadata, annotation_name, annotator_kind
    # with context.span_id as the index. AX needs them eval-namespaced.
    annotations = annotations.rename(columns={
        "label": f"eval.{eval_name}.label",
        "score": f"eval.{eval_name}.score",
        "explanation": f"eval.{eval_name}.explanation",
    })
    # Drop columns AX doesn't use; they trigger an INFO warning otherwise.
    for col in ("name", "metadata", "annotation_name", "annotator_kind"):
        if col in annotations.columns:
            annotations = annotations.drop(columns=[col])
    # AX requires string dtype for the explanation column. Code evals
    # produce no explanation, leaving an all-null column with null dtype,
    # which AX rejects ("reserved column cannot be coerced to canonical type").
    # Coerce to string so the type is stable.
    expl_col = f"eval.{eval_name}.explanation"
    if expl_col in annotations.columns:
        annotations[expl_col] = annotations[expl_col].fillna("").astype(str)
    # context.span_id must be a column, not just the index
    if annotations.index.name == "context.span_id":
        annotations = annotations.reset_index()
    arize_client.spans.update_evaluations(
        space_id=SPACE_ID,
        project_name=PROJECT_NAME,
        dataframe=annotations,
    )
    print(f"Logged {len(annotations)} {eval_name} evaluations to AX")

In [ ]:
log_eval_to_ax(results, eval_name="mentions_ticker")

  arize.spans.client | WARNING | Flight update response missing counts | project='aiewf-financial-demo' request_type='EVALUATION' total_spans=13 spans_processed=None spans_updated=13 spans_failed=None success_rate=None error_count=0


Logged 13 mentions_ticker evaluations to AX


## Step 6: Built-In Evals

Arize AX ships with pre-built evaluators for common checks. The **Correctness** evaluator uses an LLM judge to assess whether a response is factually accurate, complete, and logically consistent — no prompt engineering required.

> **Note on imports:** the evaluator classes come from `phoenix.evals` (pip package `arize-phoenix-evals`). That is Arize's open-source evaluation library — its import namespace is `phoenix.evals` whether you send data to Arize AX or to open-source Phoenix. The AX docs use these same imports.


In [ ]:
from phoenix.evals.llm import LLM
from phoenix.evals.metrics import CorrectnessEvaluator
from openinference.instrumentation import suppress_tracing

# Sonnet judges Haiku's output. Larger judge model = more reliable grading.
llm = LLM(provider="anthropic", model="claude-sonnet-4-6")

# Built-in correctness evaluator — no prompt to write
correctness_eval = CorrectnessEvaluator(llm=llm)

We suppress tracing here so it doesn't trace our evaluator itself.

In [ ]:
with suppress_tracing():
    correctness_results = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[correctness_eval]
    )

correctness_results.head()

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

,index,time,parent_id,attributes.llm.prompt_template.version,attributes.session.id,event.names,attributes.llm.cost.prompt,status_message,attributes.llm.token_count.prompt_details.cache_write,attributes.tool.id,...,attributes.exception.stacktrace,attributes.llm.token_count.prompt,event.attributes,attributes.llm.model_name,attributes.embedding.model_name,attributes.llm.token_count.completion,output,attributes.llm.token_count.completion_details.output,correctness_execution_details,correctness_score
8,8,2026-06-28 18:26:13.818000+00:00,None,,None,None,NaN,,NaN,None,...,,NaN,None,,,NaN,# TESLA INC. (NASDAQ: TSLA)\n## Financial Repo...,NaN,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
19,19,2026-06-28 18:33:30.692000+00:00,None,,None,None,NaN,,NaN,None,...,,NaN,None,,,NaN,I'll present the financial report directly for...,NaN,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
29,9,2026-06-28 18:34:34.487000+00:00,None,,None,None,NaN,,NaN,None,...,,NaN,None,,,NaN,# NVIDIA Corporation (NVDA)\n## Financial Anal...,NaN,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
38,8,2026-06-28 18:35:22.173000+00:00,None,,None,None,NaN,,NaN,None,...,,NaN,None,,,NaN,# AWS FINANCIAL REPORT\n## Amazon Web Services...,NaN,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."
55,36,2026-06-28 18:38:00.284000+00:00,None,,None,None,NaN,,NaN,None,...,,NaN,None,,,NaN,I'll create the financial report for you as a ...,NaN,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'correctness', 'score': 0.0, 'label':..."


And send the results to Arize AX again.

In [ ]:
log_eval_to_ax(correctness_results, eval_name="correctness")

  arize.spans.client | WARNING | Flight update response missing counts | project='aiewf-financial-demo' request_type='EVALUATION' total_spans=13 spans_processed=None spans_updated=13 spans_failed=None success_rate=None error_count=0


Logged 13 correctness evaluations to AX


### Built-In Eval — Faithfulness

Correctness gave us 0/13 — not because the reports are bad, but because the judge is trying to fact-check real-time financial data against its training data. That's the wrong question.

**Faithfulness** asks a better question: does the report stick to the source material the agent actually found? Instead of fact-checking against the whole internet, it checks whether the output is grounded in the research from Turn 1.

The key difference: we give the judge the agent's research as **context**.

### Extract research context from traces

Our agent works in two turns: research (web search), then write (report). The research turn's output is the context we need. We extract it from child spans and attach it to our parent spans.

In [ ]:
# Extract context: the research output from Turn 1 (child spans of our parent spans)
# Child LLM spans contain the research the agent gathered before writing

child_spans = spans_df[spans_df["parent_id"].notna()].copy()

# Get the first assistant response per trace (that's the research turn)
# Group by trace_id and take the first LLM output
research_context = {}
for trace_id, group in child_spans.groupby("context.trace_id"):
    # Find spans with LLM output (the research response)
    llm_spans = group[group["attributes.llm.output_messages"].notna()]
    if len(llm_spans) > 0:
        # Take the first LLM response as research context
        first_response = llm_spans.iloc[0]
        output = first_response.get("attributes.output.value", "")
        if output:
            research_context[trace_id] = str(output)[:5000]  # Truncate for token limits

# Map trace_id to parent span and add context column
parent_spans["context"] = parent_spans["context.trace_id"].map(research_context)
print(f"Added context to {parent_spans['context'].notna().sum()}/{len(parent_spans)} spans")
print(f"\nSample context (first 200 chars):")
first_ctx = parent_spans["context"].dropna().iloc[0] if parent_spans["context"].notna().any() else "N/A"
print(f"  {first_ctx[:200]}...")

Added context to 13/13 spans

Sample context (first 200 chars):
  ## Tesla (TSLA) Research: Financial Performance & Growth Outlook

### Financial Performance Summary

#### Q1 2026 Results
Tesla delivered solid Q1 2026 earnings with revenue reaching **$22.4 billion**...


### Run the faithfulness evaluator

FaithfulnessEvaluator needs three columns: `input`, `output`, and `context`. We have all three now.

In [ ]:
from phoenix.evals.metrics import FaithfulnessEvaluator

faithfulness_eval = FaithfulnessEvaluator(llm=llm)

# Only run on spans that have context
spans_with_context = parent_spans[parent_spans["context"].notna()].copy()

with suppress_tracing():
    faith_results = evaluate_dataframe(
        dataframe=spans_with_context,
        evaluators=[faithfulness_eval]
    )

faith_results.head()

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

,index,time,parent_id,attributes.llm.prompt_template.version,attributes.session.id,event.names,attributes.llm.cost.prompt,status_message,attributes.llm.token_count.prompt_details.cache_write,attributes.tool.id,...,attributes.llm.token_count.prompt,event.attributes,attributes.llm.model_name,attributes.embedding.model_name,attributes.llm.token_count.completion,output,attributes.llm.token_count.completion_details.output,context,faithfulness_execution_details,faithfulness_score
8,8,2026-06-28 18:26:13.818000+00:00,None,,None,None,NaN,,NaN,None,...,NaN,None,,,NaN,# TESLA INC. (NASDAQ: TSLA)\n## Financial Repo...,NaN,## Tesla (TSLA) Research: Financial Performanc...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
19,19,2026-06-28 18:33:30.692000+00:00,None,,None,None,NaN,,NaN,None,...,NaN,None,,,NaN,I'll present the financial report directly for...,NaN,## Apple (AAPL) Research Report: Revenue Growt...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 1.0, 'label'..."
29,9,2026-06-28 18:34:34.487000+00:00,None,,None,None,NaN,,NaN,None,...,NaN,None,,,NaN,# NVIDIA Corporation (NVDA)\n## Financial Anal...,NaN,## NVIDIA (NVDA) Research Report — AI Chip Dem...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."
38,8,2026-06-28 18:35:22.173000+00:00,None,,None,None,NaN,,NaN,None,...,NaN,None,,,NaN,# AWS FINANCIAL REPORT\n## Amazon Web Services...,NaN,# Amazon (AMZN) AWS Research Report\n## Perfor...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."
55,36,2026-06-28 18:38:00.284000+00:00,None,,None,None,NaN,,NaN,None,...,NaN,None,,,NaN,I'll create the financial report for you as a ...,NaN,# GOOGL Research: Advertising Revenue & AI Str...,"{'status': 'COMPLETED', 'exceptions': [], 'exe...","{'name': 'faithfulness', 'score': 0.0, 'label'..."


In [ ]:
# Compare correctness vs faithfulness results
import pandas as pd

faith_scores = pd.json_normalize(faith_results["faithfulness_score"])
print("Faithfulness results:")
print(f"  {faith_scores['label'].value_counts().to_dict()}")
print(f"\nCorrectness gave us 0/13 passes — not useful for real-time data.")
print(f"Faithfulness gives us a meaningful split — choosing the right eval matters more than tuning it.")

Faithfulness results:
  {'unfaithful': 7, 'faithful': 6}

Correctness gave us 0/13 passes — not useful for real-time data.
Faithfulness gives us a meaningful split — choosing the right eval matters more than tuning it.


In [ ]:
log_eval_to_ax(faith_results, eval_name="faithfulness")

  arize.spans.client | WARNING | Flight update response missing counts | project='aiewf-financial-demo' request_type='EVALUATION' total_spans=13 spans_processed=None spans_updated=13 spans_failed=None success_rate=None error_count=0


Logged 13 faithfulness evaluations to AX


## Step 7: Custom Eval Rubric — Actionability

Built-in evals are general-purpose. Custom rubrics check what matters for *your* application. Our financial analyst should produce actionable recommendations — not just summarize data, but tell the reader what to do.

Per the [AX evaluator docs](https://arize.com/docs/ax/evaluate/create-evaluators#tutorial-writing-a-prompt-template), a good prompt template has four parts:
1. **Define the judge's role** — domain context: what system it evaluates, what domain, what its task is
2. **Explicit criteria** — measurable pass/fail conditions, including failure modes from your error analysis
3. **Label the data with XML tags** — wrap each `{variable}` in clear XML tags so the judge knows where instructions end and data begins
4. **Leave output format out of the prompt** — don't tell the judge what labels to emit; the possible responses are defined externally as the evaluator's **Choices** (the `choices=` arg below)

We also add **labeled examples** of each class — not required by the docs, but they measurably improve judge consistency, so we keep them.

In [ ]:
actionability_template = """
You are an expert financial analyst evaluator. Your task is to judge whether
a financial report provides actionable investment guidance, not just raw data.

ACTIONABLE — The report:
- Contains specific recommendations (buy/sell/hold or equivalent guidance)
- Identifies concrete risks with supporting data
- Includes forward-looking analysis, not just historical data
- Provides context for WHY recommendations are made

NOT ACTIONABLE — The report:
- Only summarizes publicly available data without interpretation
- Lacks specific recommendations or next steps
- Presents risks without supporting evidence
- Contains only backward-looking analysis

Here are examples of each:

Example — ACTIONABLE:
\"Based on NVDA's 122% YoY revenue growth driven by data center demand,
strong forward P/E of 35x relative to sector median of 22x, and expanding
margins, NVDA presents a compelling growth position. Key risk: concentration
in AI training chips (~70% of revenue). Recommendation: accumulate on
pullbacks below $800.\"

Example — NOT ACTIONABLE:
\"NVDA is a major player in the semiconductor industry. The company has seen
significant growth in recent years driven by AI demand. NVDA's stock has
performed well. Investors should consider various factors when making
investment decisions.\"

<user_query>
{input}
</user_query>

<financial_report>
{output}
</financial_report>
"""

### Online path

1. Click Online Evaluators -> Add Evaluator
2. Select LLM as a judge evaluator
3. Create an LLM configuration with your Anthropic API key
4. Give it a name
5. Set scope to Trace
6. Paste in the rubric above
7. Set labels to "actionable" (1) and "not actionable" (0)
8. Under matching traces select "single trace"
9. Map input to  input and output to output
10. Save the evaluator to the hub
11. Click Run


### Offline path

Run our new custom judge.

In [ ]:
from phoenix.evals import ClassificationEvaluator

# ClassificationEvaluator builds a custom LLM-as-judge evaluator
# from a prompt template. The labels live in `choices`, not the prompt.
actionability_evaluator = ClassificationEvaluator(
    name="actionability",
    llm=llm,
    prompt_template=actionability_template,
    choices={"actionable": 1.0, "not actionable": 0.0},
)

with suppress_tracing():
    action_results_df = evaluate_dataframe(
        dataframe=parent_spans, evaluators=[actionability_evaluator]
    )

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
import pandas as pd

pd.json_normalize(action_results_df["actionability_score"].head())

,name,score,label,explanation,kind,direction,metadata.model
0,actionability,1.0,actionable,The report is actionable for several key reaso...,llm,maximize,claude-sonnet-4-6
1,actionability,0.0,not actionable,"The report provides detailed financial data, f...",llm,maximize,claude-sonnet-4-6
2,actionability,1.0,actionable,This financial report is actionable for severa...,llm,maximize,claude-sonnet-4-6
3,actionability,0.0,not actionable,The report provides strong quantitative data a...,llm,maximize,claude-sonnet-4-6
4,actionability,1.0,actionable,The report provides a clear investment rating ...,llm,maximize,claude-sonnet-4-6


Log actionability results back to Arize AX


In [ ]:
log_eval_to_ax(action_results_df, eval_name="actionability")

  arize.spans.client | WARNING | Flight update response missing counts | project='aiewf-financial-demo' request_type='EVALUATION' total_spans=13 spans_processed=None spans_updated=13 spans_failed=None success_rate=None error_count=0


Logged 13 actionability evaluations to AX


## Step 8: Meta-Evaluation — Can You Trust Your Judge?

Your LLM judge is a classifier. It makes predictions (actionable / not actionable) that you can check against ground truth — your own human judgment. This is **meta-evaluation**: testing the test.

We'll:
1. **Label a handful of examples in AX** — read the output, apply an annotation in the UI
2. **Pull the labels back** — annotations come back as columns in the span export
3. **Run the judge on the same examples** — see what it says
4. **Compare** — where do they agree? Where do they disagree?
5. **Calculate precision and recall** — put numbers on judge quality

### Label some examples in AX

The easiest place to apply human labels is in the AX UI. AX calls them **annotations**.

**One-time setup** — create an annotation config:
1. Open any trace from the `aiewf-financial-demo` project.
2. Click **Add Annotation** → **+ New Annotation**.
3. Name it `human_actionable`, type **Categorical**, values `actionable` and `not actionable`.

**Then label a handful of examples** (aim for 6+):
- Read the input and the output.
- Apply `actionable` or `not actionable` using the same criteria your rubric uses.
- If you find yourself hesitating, the rubric is ambiguous — note what made you hesitate.

When you've labeled some spans, run the next cell to pull the annotations back.

In [ ]:
# Re-pull spans, this time bringing back any annotations applied in the AX UI.
# Annotations arrive as `annotation.<name>.label` columns.
spans_df = arize_client.spans.export_to_df(
    space_id=SPACE_ID,
    project_name=PROJECT_NAME,
    start_time=datetime.utcnow() - timedelta(days=7),
    end_time=datetime.utcnow(),
)

# Same parent-span filter we used in Step 4
parent_spans = spans_df[spans_df["parent_id"].isna()].copy()

if "input" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.input.value": "input"}, inplace=True)
if "output" not in parent_spans.columns:
    parent_spans.rename(columns={"attributes.output.value": "output"}, inplace=True)
parent_spans = parent_spans.loc[:, ~parent_spans.columns.duplicated()]

ANNOTATION_COL = "annotation.human_actionable.label"
if ANNOTATION_COL not in parent_spans.columns:
    raise RuntimeError(
        f"No '{ANNOTATION_COL}' column in the export. "
        "Have you applied any human_actionable annotations in the AX UI yet?"
    )

labeled_subset = parent_spans[parent_spans[ANNOTATION_COL].notna()].copy()
print(f"Found {len(labeled_subset)} labeled spans.")
display(labeled_subset[["input", ANNOTATION_COL]])

  arize._exporter.client | INFO | Fetching data...


/tmp/ipykernel_3840/2595012529.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow() - timedelta(days=7),
/tmp/ipykernel_3840/2595012529.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time=datetime.utcnow(),
INFO:arize._exporter.client:Fetching data...


  exporting 207 rows:   0%|                             | 0/207 [00:00, ? row/s]

  exporting 207 rows: 100%|█████████████████████| 207/207 [00:00, 1387.96 row/s]

Found 13 labeled spans.


,input,annotation.human_actionable.label
8,Research: TSLA\nFocus: financial performance a...,actionable
19,Research: AAPL\nFocus: revenue growth and serv...,actionable
29,Research: NVDA\nFocus: AI chip demand and valu...,actionable
38,Research: AMZN\nFocus: AWS performance and pro...,actionable
55,Research: GOOGL\nFocus: advertising revenue an...,not actionable
80,Research: MSFT\nFocus: cloud computing segment,not actionable
100,Research: META\nFocus: metaverse investments a...,not actionable
114,Research: TSLA\nFocus: vehicle deliveries and ...,not actionable
138,Research: RIVN\nFocus: financial health and fu...,actionable
162,"Research: AAPL, MSFT\nFocus: comparative finan...",actionable


### Run the judge on the same examples

Now run the actionability evaluator on exactly the same traces. Same inputs, same outputs — the only question is whether the judge agrees with you.

In [ ]:
# Run our actionability judge on the labeled subset
with suppress_tracing():
    judge_results = evaluate_dataframe(
        dataframe=labeled_subset, evaluators=[actionability_evaluator]
    )

# Extract judge labels
judge_labels = pd.json_normalize(judge_results["actionability_score"])
labeled_subset["judge_label"] = judge_labels["label"].values
labeled_subset["judge_explanation"] = judge_labels["explanation"].values

Evaluating Dataframe |          | 0/13 (0.0%) | ⏳ 00:00<? | ?it/s

### Compare — where do they agree and disagree?

When the judge disagrees with you, read its explanation. Often it reveals ambiguity in your rubric — that's the most valuable insight from meta-evaluation.

In [ ]:
# Side-by-side comparison: human annotation vs judge prediction
comparison = labeled_subset[["input", ANNOTATION_COL, "judge_label"]].copy()
comparison = comparison.rename(columns={ANNOTATION_COL: "human_label"})
# Normalize the underscore convention — UI annotations use "not_actionable", judge prompt uses "not actionable".
comparison["human_label"] = comparison["human_label"].str.replace("_", " ")
comparison["agree"] = comparison["human_label"] == comparison["judge_label"]

print("Human vs. Judge Comparison")
print("=" * 70)
for i, (idx, row) in enumerate(comparison.iterrows()):
    agree_mark = "✓" if row["agree"] else "✗ DISAGREE"
    print(f"  {row['input'][:45]:<45}  human: {row['human_label']:<15} judge: {row['judge_label']:<15} {agree_mark}")

agreement_rate = comparison["agree"].mean()
print(f"\nAgreement rate: {agreement_rate:.0%} ({comparison['agree'].sum()}/{len(comparison)})")

# Show explanations for disagreements
disagreements = labeled_subset[comparison["human_label"].values != labeled_subset["judge_label"].values]
if len(disagreements) > 0:
    print(f"\n{'='*70}")
    print("DISAGREEMENTS — read the judge's reasoning:")
    for idx, row in disagreements.iterrows():
        human = str(row[ANNOTATION_COL]).replace("_", " ")
        print(f"\n  Query: {row['input'][:60]}")
        print(f"  Human said: {human}")
        print(f"  Judge said: {row['judge_label']}")
        print(f"  Judge's reasoning: {row['judge_explanation'][:300]}...")
else:
    print("\nNo disagreements! The judge matches your labels perfectly.")

Human vs. Judge Comparison
  Research: TSLA
Focus: financial performance a  human: actionable      judge: actionable      ✓
  Research: AAPL
Focus: revenue growth and serv  human: actionable      judge: not actionable  ✗ DISAGREE
  Research: NVDA
Focus: AI chip demand and valu  human: actionable      judge: actionable      ✓
  Research: AMZN
Focus: AWS performance and pro  human: actionable      judge: not actionable  ✗ DISAGREE
  Research: GOOGL
Focus: advertising revenue an  human: not actionable  judge: actionable      ✗ DISAGREE
  Research: MSFT
Focus: cloud computing segment  human: not actionable  judge: not actionable  ✓
  Research: META
Focus: metaverse investments a  human: not actionable  judge: not actionable  ✓
  Research: TSLA
Focus: vehicle deliveries and   human: not actionable  judge: actionable      ✗ DISAGREE
  Research: RIVN
Focus: financial health and fu  human: actionable      judge: actionable      ✓
  Research: AAPL, MSFT
Focus: comparative finan  human: actionab

### Precision and recall

- **Precision**: When the judge says "not actionable," is it right?
- **Recall**: Of all reports that are truly not actionable, how many does the judge catch?

In most eval scenarios, prioritize **recall** — it's better to flag false positives than to miss real failures.

In [ ]:
# Simple precision/recall calculation for the "not actionable" (fail) class.
# We care most about catching failures, so "not actionable" is our positive class.

human_fail = set(comparison[comparison["human_label"] == "not actionable"].index)
judge_fail = set(comparison[comparison["judge_label"] == "not actionable"].index)

true_positives = len(human_fail & judge_fail)
false_positives = len(judge_fail - human_fail)
false_negatives = len(human_fail - judge_fail)
true_negatives = len(comparison) - true_positives - false_positives - false_negatives

print("Confusion Matrix (for 'not actionable' class)")
print("=" * 45)
print(f"                    Judge: fail  Judge: pass")
print(f"  Human: fail           {true_positives}            {false_negatives}")
print(f"  Human: pass           {false_positives}            {true_negatives}")
print()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0

print(f"Precision: {precision:.0%}  (when the judge says 'fail', is it right?)")
print(f"Recall:    {recall:.0%}  (of all real fails, how many does it catch?)")
print(f"\nWith only {len(comparison)} examples, these numbers are rough.")
print(f"For a proper validation, aim for 20-50 labeled examples.")

Confusion Matrix (for 'not actionable' class)
                    Judge: fail  Judge: pass
  Human: fail           2            4
  Human: pass           3            4

Precision: 40%  (when the judge says 'fail', is it right?)
Recall:    33%  (of all real fails, how many does it catch?)

With only 13 examples, these numbers are rough.
For a proper validation, aim for 20-50 labeled examples.


## Step 9: Improve the Agent — Automatically

The actionability eval told us *what's wrong*, and its explanations told us *why*. Instead of hand-editing the prompts ourselves, we feed those explanations straight back to Claude and let it rewrite the agent's prompts — then run an experiment to prove the new prompts are actually better.

**The loop:**
1. Collect the judge explanations from every failing trace.
2. Hand Claude the current prompts + those explanations + our requirements.
3. Claude returns revised prompts.
4. Run an experiment on the failure dataset — did the scores climb?

Before the experiment cell, save the failure dataset: in the AX UI, filter traces to `actionability == "not actionable"`, click **Save as Dataset**, and name it `aiewf-financial-demo-fails`.

In [ ]:
# Collect the judge explanations from every failing trace, then build a
# single prompt that asks Claude to rewrite the agent's prompts.
# Step 7 already scored every trace — reuse action_results_df.
import pandas as pd

scores = pd.json_normalize(action_results_df["actionability_score"])
parent_spans["actionability_label"] = scores["label"].values
parent_spans["actionability_explanation"] = scores["explanation"].values

failing = parent_spans[parent_spans["actionability_label"] == "not actionable"].copy()
print(f"{len(failing)} of {len(parent_spans)} traces failed the actionability eval\n")

REQUIREMENTS = """The financial analysis agent must produce reports that:
- Reference the correct ticker(s) requested by the user
- Include real, recent financial data (ratios, prices, recent news)
- Distinguish forward-looking analysis from historical summary
- Provide actionable recommendations with explicit buy/sell/hold language
- Identify concrete risks with supporting data
- For multi-ticker queries, include a dedicated comparison section"""

explanations = "\n".join(
    f"- {row['actionability_explanation']}" for _, row in failing.iterrows()
)

improvement_prompt = f"""You are improving the prompts of a financial-analysis agent.
The agent runs in two turns: a RESEARCH turn and a WRITE turn.

An LLM judge scored the agent's reports for "actionability" and explained
every failure. Find the RECURRING THEMES across these explanations and
rewrite the two prompts to address them. Keep the changes faithful to the
REQUIREMENTS - improve the product, not just the eval score.

## CURRENT RESEARCH PROMPT
{RESEARCH_PROMPT}

## CURRENT WRITE PROMPT
{WRITE_PROMPT}

## REQUIREMENTS
{REQUIREMENTS}

## JUDGE EXPLANATIONS FOR FAILING REPORTS
{explanations}

Return the two revised prompts, each wrapped in XML tags exactly like this:
<research_prompt>...revised research prompt...</research_prompt>
<write_prompt>...revised write prompt...</write_prompt>
Keep the {{tickers}} and {{focus}} placeholders in the research prompt."""

print(improvement_prompt)

6 of 13 traces failed the actionability eval

You are improving the prompts of a financial-analysis agent.
The agent runs in two turns: a RESEARCH turn and a WRITE turn.

An LLM judge scored the agent's reports for "actionability" and explained
every failure. Find the RECURRING THEMES across these explanations and
rewrite the two prompts to address them. Keep the changes faithful to the
REQUIREMENTS - improve the product, not just the eval score.

## CURRENT RESEARCH PROMPT
Research {tickers}. Focus on: {focus}.
Use web search to find current financial data, news, and trends.

## CURRENT WRITE PROMPT
Now write a concise financial report based on your research above.

## REQUIREMENTS
The financial analysis agent must produce reports that:
- Reference the correct ticker(s) requested by the user
- Include real, recent financial data (ratios, prices, recent news)
- Distinguish forward-looking analysis from historical summary
- Provide actionable recommendations with explicit buy/sell/hold 

### Ask Claude to rewrite the prompts

We send that prompt to Claude with the `anthropic` SDK — the same package the judge uses — and parse the two revised prompts out of the response. The call is wrapped in `suppress_tracing()` so this meta-step doesn't show up as a trace of the *application*.

In [ ]:
import re
import anthropic

anthropic_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

with suppress_tracing():
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        messages=[{"role": "user", "content": improvement_prompt}],
    )
reply = response.content[0].text

IMPROVED_RESEARCH_PROMPT = re.search(
    r"<research_prompt>(.*?)</research_prompt>", reply, re.DOTALL
).group(1).strip()
IMPROVED_WRITE_PROMPT = re.search(
    r"<write_prompt>(.*?)</write_prompt>", reply, re.DOTALL
).group(1).strip()

print("=== IMPROVED RESEARCH PROMPT ===")
print(IMPROVED_RESEARCH_PROMPT)
print("\n=== IMPROVED WRITE PROMPT ===")
print(IMPROVED_WRITE_PROMPT)

=== IMPROVED RESEARCH PROMPT ===
Research {tickers}. Focus on: {focus}.

Use web search to find current financial data, news, and trends. Your research MUST gather the following — gaps here will produce an unusable report:

**Financials & Valuation**
- Current price, 52-week range, and YTD performance
- Key ratios: P/E (trailing & forward), EV/EBITDA, P/S, PEG — and how they compare to sector peers or historical averages
- Recent earnings: revenue, margins, EPS (actual vs. consensus), and any guidance issued by management

**Forward-Looking Data**
- Management guidance for next quarter and full fiscal year
- Analyst consensus estimates (revenue, EPS) and any recent estimate revisions
- Published analyst price targets (low / consensus / high) and the firms behind them

**Growth Catalysts**
- Specific, near-term drivers with supporting data (product launches, market share trends, macro tailwinds, margin expansion levers)

**Risks — with supporting evidence**
- Regulatory, competitive, ma

### Wire up the improved agent

Same two-turn agent as Step 2 — it just uses Claude's revised prompts now. We use `.replace()` rather than `.format()` for the placeholders, since an LLM-written prompt may contain other braces.

In [ ]:
async def improved_financial_report(tickers: str, focus: str) -> str:
    with tracer.start_as_current_span(
        "financial_report",
        attributes={
            "input.value": f"Research: {tickers}\nFocus: {focus}",
        },
    ) as span:
        async with ClaudeSDKClient(options=options) as client:
            # Turn 1: Research (Claude's improved prompt)
            research = (
                IMPROVED_RESEARCH_PROMPT
                .replace("{tickers}", tickers)
                .replace("{focus}", focus)
            )
            await client.query(research)
            async for message in client.receive_response():
                pass  # Let the research complete

            # Turn 2: Write report (Claude's improved prompt)
            await client.query(IMPROVED_WRITE_PROMPT)
            report = ""
            async for message in client.receive_response():
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            report += block.text
                elif isinstance(message, ResultMessage):
                    report = message.result or report

            span.set_attribute("output.value", report)
    tracer_provider.force_flush()
    return report

### Define the task and the evaluator

The **task** runs the improved agent on one dataset example. The **evaluator** scores the task's output — we reuse the actionability rubric from Step 7, wrapped so it returns an `EvaluationResult`, the type AX experiments expect.

In [ ]:
import asyncio
import concurrent.futures

# EvaluationResult is the type an experiment evaluator must return.
# The import path has moved across SDK versions, so try both.
try:
    from arize.experiments import EvaluationResult
except ImportError:
    from arize.experimental.datasets.experiments.evaluators.base import EvaluationResult


def run_async(coro):
    """Run an async coroutine to completion, even from inside the notebook's
    already-running event loop, by giving it a fresh loop in a worker thread."""
    with concurrent.futures.ThreadPoolExecutor(1) as pool:
        return pool.submit(lambda: asyncio.run(coro)).result()


def improved_agent_task(dataset_row):
    """Task: run the improved agent on one dataset example.
    Each example's input looks like 'Research: TSLA\nFocus: ...'."""
    inp = dataset_row.get("attributes.input.value", "")
    lines = inp.split("\n")
    tickers = lines[0].replace("Research: ", "").strip()
    focus = lines[1].replace("Focus: ", "").strip() if len(lines) > 1 else ""
    return run_async(improved_financial_report(tickers, focus))


def actionability_eval(output, dataset_row):
    """Evaluator: score the task output with the Step 7 actionability rubric.
    Wraps the ClassificationEvaluator so it returns an EvaluationResult."""
    inp = dataset_row.get("attributes.input.value", "")
    scores = actionability_evaluator.evaluate({"input": inp, "output": output})
    s = scores[0]
    return EvaluationResult(
        score=float(s.score) if s.score is not None else 0.0,
        label=s.label,
        explanation=getattr(s, "explanation", "") or "",
    )

### Run the experiment

`experiments.run` runs the task on every example in the dataset, scores each output with the evaluators, and stores the run in AX. The `dataset` argument takes the dataset name you saved it under in the UI.

In [ ]:
# Run the experiment. AX runs improved_agent_task on every example in the
# dataset, scores each output with actionability_eval, and stores the run.
# A dataset *name* needs `space` to resolve; pass a dataset ID instead to skip it.
experiment, experiment_df = arize_client.experiments.run(
    name="improved-prompts-v1",
    dataset="aiewf-financial-demo-fails",   # dataset name saved from the AX UI
    space=SPACE_ID,
    task=improved_agent_task,
    evaluators=[actionability_eval],
    concurrency=2,
)

print(f"Experiment complete: {len(experiment_df)} examples scored.")
experiment_df

  arize.experiments.functions | INFO | 🧪 Experiment started.


INFO:arize.experiments.functions:🧪 Experiment started.


  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (06/28/26 07:45 PM +0000)
---------------------------------------
|   n_examples |   n_runs |   n_errors |
|-------------:|---------:|-----------:|
|            6 |        6 |          0 |


...

  arize.experiments.functions | INFO | ✅ All evaluators completed.


INFO:arize.experiments.functions:✅ All evaluators completed.


  arize.pre_releases | WARNING | [BETA] experiments.get is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


Experiment complete: 6 examples scored.


,example_id,output,error,result.trace.id,result.trace.timestamp,eval.actionability_eval.score,eval.actionability_eval.label,eval.actionability_eval.explanation,eval.actionability_eval.trace.id,eval.actionability_eval.trace.timestamp
0,029535c6-5b67-41b2-bf96-a8bebec40b66,## MSFT Financial Report\n\n### 1. Snapshot\n\...,None,9c37bcf5feea6c0d0bc18af692757081,1782674975492,1.0,actionable,This financial report is clearly actionable. I...,856cc024d8299f3785d387b4dba117e2,1782675906698
1,03909a55-eabd-4b2f-b1b7-333855708791,---\n\n## AAPL Financial Report\n\n### 1. Snap...,None,d6da106484f11b6c9de75aa653bc38e1,1782675159873,1.0,actionable,The report is clearly actionable. It contains ...,c48fec4714db9c75f7036fb6868d02fc,1782675915484
2,4f98b64c-0c25-427c-930f-c1336f78f549,---\n\n## TSLA Financial Report\n\n### 1. Snap...,None,8d2155ecf85acf10c263f0fa35034cf7,1782675301906,1.0,actionable,This financial report is clearly actionable. I...,9580157b162052c2ad1744aefd6c7b1d,1782675922452
3,017474e1-8c91-4263-b1dc-f1f11191ba32,---\n\n## AMZN Financial Report\n\n### 1. Snap...,None,17ac90c7838d395304e4b40ad90ee7ec,1782675414923,1.0,actionable,This report is clearly actionable. It provides...,c2263b53d02175e5f8ec83b12b518299,1782675928550
4,32d303ad-267a-4be2-9ea1-af17b933edf9,## META Financial Report\n\n### 1. Snapshot\n\...,None,e9c181bea9b3afa89f5bd49ffa571099,1782675560512,1.0,actionable,This report is clearly actionable. It contains...,b79b66c44b762e520a789f5b13c06cec,1782675935667
5,eb4656fd-fdf7-4557-9e7f-b69a7e6bb74a,## AMZN Financial Report\n\n### 1. Snapshot\n*...,None,33b1eadc308d414c1947c88893e0dcd9,1782675756129,1.0,actionable,This financial report is clearly actionable. I...,9c764c6ced9ea3f3e56db2e655d8cb86,1782675942059


### Read the results

Each row of `experiment_df` is one dataset example: the improved agent's output and its actionability score. Open the dataset's **Experiments** view in the AX UI to compare this run against earlier ones, example by example — that side-by-side is how you prove Claude's prompt rewrite actually helped.

That's the whole loop: trace → eval → explanation → Claude rewrites the prompt → experiment proves it. The evals didn't just tell you the agent was broken; they drove the fix.